# PLDStega Kaggle GPU Notebook

Use this notebook on Kaggle with GPU enabled and Internet enabled. Upload this repository as a Kaggle Dataset, or upload a zip of the repository as a Kaggle Dataset. The notebook copies the repo into `/kaggle/working`, installs dependencies, runs CPU-safe tests, then runs a PLDStega SDXL hide/extract smoke test.

This is an experimental research prototype. The extract step uses only the stego image and key, but decoding parameters such as image size, capacity, repetition, ECC symbols, and embed method must match the hide step.

## 1. Kaggle Setup Checklist

Before running:

- In Kaggle notebook settings, set `Accelerator` to a GPU.
- Turn `Internet` on so Diffusers can download SDXL.
- Add this project folder or zip as a Kaggle Dataset.
- Optional: add a Kaggle secret named `HF_TOKEN` if Hugging Face rate limits or model access require login.

In [ ]:
from pathlib import Path
import os
import shutil
import sys
import zipfile

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
PROJECT_NAME = 'LDStega_implementation'
PROJECT_DIR = WORK_DIR / PROJECT_NAME

def find_repo_root(base: Path):
    for marker in base.rglob('ldstega'):
        if marker.is_dir() and (marker.parent / 'README.md').exists():
            return marker.parent
    return None

repo_root = find_repo_root(INPUT_DIR)

if repo_root is None:
    zips = sorted(INPUT_DIR.rglob('*.zip'))
    if not zips:
        raise FileNotFoundError(
            'Could not find the repo under /kaggle/input. Upload the project folder or a zip as a Kaggle Dataset.'
        )
    unzip_dir = WORK_DIR / 'uploaded_repo_unzipped'
    unzip_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0], 'r') as zf:
        zf.extractall(unzip_dir)
    repo_root = find_repo_root(unzip_dir)
    if repo_root is None:
        raise FileNotFoundError(f'Zip did not contain a repo with ldstega/ and README.md: {zips[0]}')

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
shutil.copytree(repo_root, PROJECT_DIR)
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print('Project copied to:', PROJECT_DIR)
print('Repo files:', sorted(p.name for p in PROJECT_DIR.iterdir())[:20])

## 2. Install Runtime Dependencies

Kaggle sometimes assigns older Tesla P100 GPUs. Newer Kaggle PyTorch builds may not include CUDA kernels for P100 (`sm_60`), which causes `CUDA error: no kernel image is available for execution on the device`. The next cell checks the GPU before importing `torch` and installs a compatible PyTorch, TorchVision, TorchAudio, Diffusers, and Transformers set when needed.

In [ ]:
import subprocess
import sys

def get_gpu_info_without_torch():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
            text=True,
        ).strip()
    except Exception as exc:
        raise RuntimeError('Could not query GPU with nvidia-smi. Enable Kaggle GPU first.') from exc
    name, compute_cap = [part.strip() for part in out.split(',', 1)]
    return name, compute_cap

gpu_name, compute_cap = get_gpu_info_without_torch()
major = int(compute_cap.split('.')[0])
print('GPU:', gpu_name)
print('Compute capability:', compute_cap)

if major < 7:
    print('Installing PyTorch 2.4.1 CUDA 12.1 wheels for older GPU support such as Tesla P100 sm_60...')
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir',
            'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
            '--index-url', 'https://download.pytorch.org/whl/cu121',
        ],
        check=True,
    )
else:
    print('Current Kaggle PyTorch build should support this GPU generation.')

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', '--no-deps',
        'diffusers==0.30.3',
        'transformers==4.44.2',
        'accelerate==0.33.0',
        'safetensors==0.4.5',
        'huggingface-hub==0.24.7',
        'numpy<2.1',
        'pillow<12',
        'cryptography',
        'reedsolo',
    ],
    check=True,
)

In [ ]:
check_code = r"""
import os
import sys
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available. Enable Accelerator = GPU before running SDXL.')
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA capability:', torch.cuda.get_device_capability(0))
print('Total VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
try:
    import torchaudio
    print('Torchaudio:', torchaudio.__version__)
except Exception as exc:
    raise RuntimeError(f'torchaudio import failed. Re-run install cell from a fresh Kaggle session: {exc}')
import diffusers
import transformers
print('Diffusers:', diffusers.__version__)
print('Transformers:', transformers.__version__)
major, minor = torch.cuda.get_device_capability(0)
if major < 7 and not torch.__version__.startswith('2.4.1'):
    raise RuntimeError(
        'P100 requires the notebook install cell to downgrade torch to 2.4.1. '
        'Restart the Kaggle session, then run from cell 1 again.'
    )
"""
subprocess.run([sys.executable, '-c', check_code], check=True)

## 3. Optional Hugging Face Login

This is optional, but useful for rate limits and gated model access. Add a Kaggle secret named `HF_TOKEN` if needed.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = None

if token:
    from huggingface_hub import login
    login(token=token)
    print('Logged in to Hugging Face using Kaggle secret HF_TOKEN.')
else:
    print('No HF_TOKEN secret found. Continuing unauthenticated.')

## 4. Run CPU-Safe Tests

These tests do not generate SDXL images. The GPU integration test is skipped unless explicitly enabled by the test file.

In [ ]:
!python -m unittest discover -s tests -v

## 5. Configure PLDStega Smoke Test

Default settings are conservative for 8 GB GPUs. If Kaggle gives you a T4 and it runs out of memory, keep `USE_CPU_OFFLOAD=True`, or reduce `HEIGHT` and `WIDTH` to `512` for debugging.

In [ ]:
MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
PROMPT = 'realistic photo of a boy sitting on a chair playing an acoustic guitar, natural indoor light, sharp focus'
NEGATIVE_PROMPT = 'blurry, distorted, deformed hands, extra fingers, low quality'
MESSAGE = 'KOUSHIK'
KEY = 'shared-key'
SEED = 1234

HEIGHT = 768
WIDTH = 768
STEPS = 20
GUIDANCE_SCALE = 7.5

CAPACITY_BYTES = 128
GROUP_SIZE = 5
REPEAT = 3
ECC_SYMBOLS = 32
EMBED_METHOD = 'sign'
EMBED_STRENGTH = 0.05

USE_CPU_OFFLOAD = True
OUTPUT = 'outputs/pldstega_smoke.png'

Path('outputs').mkdir(exist_ok=True)
print('Output:', OUTPUT)

## 6. Hide Message During SDXL Generation

This is the heavy cell. It downloads SDXL on first run and generates the stego image.

In [ ]:
import os
import subprocess

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

hide_cmd = [
    sys.executable, '-m', 'ldstega.cli', 'hide',
    '--mode', 'pldstega',
    '--model', MODEL,
    '--prompt', PROMPT,
    '--negative-prompt', NEGATIVE_PROMPT,
    '--message', MESSAGE,
    '--key', KEY,
    '--seed', str(SEED),
    '--output', OUTPUT,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--steps', str(STEPS),
    '--guidance-scale', str(GUIDANCE_SCALE),
    '--capacity-bytes', str(CAPACITY_BYTES),
    '--group-size', str(GROUP_SIZE),
    '--repeat', str(REPEAT),
    '--ecc-symbols', str(ECC_SYMBOLS),
    '--embed-method', EMBED_METHOD,
    '--embed-strength', str(EMBED_STRENGTH),
]
if USE_CPU_OFFLOAD:
    hide_cmd.append('--cpu-offload')

print(' '.join(hide_cmd))
subprocess.run(hide_cmd, check=True, env=os.environ.copy())

In [ ]:
from IPython.display import Image as DisplayImage, display

display(DisplayImage(filename=OUTPUT))

## 7. Promptless Extract

This command intentionally does not pass prompt, seed, scheduler, steps, or guidance scale. It uses only image + key plus decoding parameters.

In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

extract_cmd = [
    sys.executable, '-m', 'ldstega.cli', 'extract',
    '--mode', 'pldstega',
    '--image', OUTPUT,
    '--key', KEY,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--capacity-bytes', str(CAPACITY_BYTES),
    '--group-size', str(GROUP_SIZE),
    '--repeat', str(REPEAT),
    '--ecc-symbols', str(ECC_SYMBOLS),
    '--embed-method', EMBED_METHOD,
]

print(' '.join(extract_cmd))
result = subprocess.run(extract_cmd, check=True, capture_output=True, text=True, env=os.environ.copy())
print('Recovered message:')
print(result.stdout.strip())

## 8. Optional Robustness Smoke Checks

These are not benchmark claims. They are quick checks to see whether the current parameters survive simple edits. Keep disabled until the PNG round trip works.

In [ ]:
RUN_ROBUSTNESS = False

if RUN_ROBUSTNESS:
    from PIL import Image
    img = Image.open(OUTPUT).convert('RGB')
    jpeg_path = 'outputs/pldstega_smoke_q90.jpg'
    img.save(jpeg_path, quality=90)
    cmd = extract_cmd.copy()
    cmd[cmd.index('--image') + 1] = jpeg_path
    print(' '.join(cmd))
    result = subprocess.run(cmd, check=True, capture_output=True, text=True, env=os.environ.copy())
    print('Recovered from JPEG Q90:')
    print(result.stdout.strip())
else:
    print('Robustness smoke checks are disabled.')

## 9. Download Outputs

Kaggle stores generated files under `/kaggle/working`. The image is also visible in the notebook output panel.

In [ ]:
from IPython.display import FileLink

display(FileLink(OUTPUT))